# Module 01 — CPU vs GPU: Matrix Multiplication Benchmark

This notebook is the interactive version of `inspect_and_benchmark.py`. Run it on a Lightning AI Studio with one NVIDIA T4.

We will inspect the GPU, compare CPU and GPU GEMM timing, measure correctness, inspect memory usage, and optionally include CPU-to-GPU transfer overhead.

## Experiment controls

Keep the inputs resident on the GPU for the compute-only comparison. The transfer-inclusive measurement deliberately includes copying inputs to the GPU and the result back to the CPU.

In [ ]:
import statistics
import time

import torch

# Change these controls to trade experiment length for more stable medians.
sizes = [128, 512, 1024, 2048]
repeats = 30
include_transfer = True
dtype = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"Matrix sizes: {sizes}")
print(f"Repeats: {repeats}")

## Inspect the environment

In [ ]:
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA runtime: {torch.version.cuda}")

if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(
            f"GPU {index}: {props.name} | compute capability "
            f"{props.major}.{props.minor} | VRAM {props.total_memory / 2**30:.2f} GiB"
        )
else:
    print("No CUDA device found; GPU measurements will be skipped.")

## Timing helper

GPU work is asynchronous relative to Python. CUDA events measure elapsed device time without confusing kernel execution with CPU launch time.

In [ ]:
def median_seconds(fn, repeats, device):
    """Warm up and return median elapsed seconds plus the final result."""
    # The warm-up absorbs first-use costs such as kernel selection and
    # library initialization before the repeated samples are collected.
    result = fn()
    if device.type == "cuda":
        torch.cuda.synchronize(device)

    samples = []
    for _ in range(repeats):
        if device.type == "cuda":
            # CUDA events measure elapsed device time on the active stream.
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            result = fn()
            end.record()
            end.synchronize()
            samples.append(start.elapsed_time(end) / 1000.0)
        else:
            # CPU work is synchronous, so a host timer is sufficient here.
            start_time = time.perf_counter()
            result = fn()
            samples.append(time.perf_counter() - start_time)
    return statistics.median(samples), result

## Benchmark one matrix size

In [ ]:
def benchmark(size, repeats, device, include_transfer):
    # Use the same reproducible inputs for both devices so max_error is
    # a meaningful correctness check.
    generator = torch.Generator(device="cpu").manual_seed(17)
    a_cpu = torch.randn((size, size), generator=generator, dtype=dtype)
    b_cpu = torch.randn((size, size), generator=generator, dtype=dtype)

    cpu_time, cpu_result = median_seconds(lambda: a_cpu @ b_cpu, repeats, torch.device("cpu"))
    row = {
        "size": size,
        "cpu_ms": cpu_time * 1000,
        "gpu_ms": None,
        "speedup": None,
        "gpu_tflops": None,
        "transfer_ms": None,
        "max_error": None,
    }

    if device.type != "cuda":
        return row

    # Resident-compute timing starts after inputs have been copied to GPU.
    a = a_cpu.to(device)
    b = b_cpu.to(device)
    torch.cuda.reset_peak_memory_stats(device)
    gpu_time, gpu_result = median_seconds(lambda: a @ b, repeats, device)
    row["gpu_ms"] = gpu_time * 1000
    row["speedup"] = cpu_time / gpu_time
    row["gpu_tflops"] = 2 * size**3 / gpu_time / 1e12
    row["max_error"] = (cpu_result.to(device) - gpu_result).abs().max().item()
    row["allocated_mib"] = torch.cuda.memory_allocated(device) / 2**20
    row["reserved_mib"] = torch.cuda.memory_reserved(device) / 2**20
    row["peak_mib"] = torch.cuda.max_memory_allocated(device) / 2**20

    if include_transfer:
        # This end-to-end path includes host-to-device copies, GEMM, and
        # device-to-host copy-back, unlike the GPU-resident timing above.
        transfer_samples = []
        for _ in range(repeats):
            start_time = time.perf_counter()
            a_roundtrip = a_cpu.to(device)
            b_roundtrip = b_cpu.to(device)
            result_roundtrip = (a_roundtrip @ b_roundtrip).cpu()
            torch.cuda.synchronize(device)
            del a_roundtrip, b_roundtrip, result_roundtrip
            transfer_samples.append(time.perf_counter() - start_time)
        row["transfer_ms"] = statistics.median(transfer_samples) * 1000

    return row

## Run the experiment

In [ ]:
results = [benchmark(size, repeats, device, include_transfer) for size in sizes]

for row in results:
    message = (
        f"N={row['size']:5d} | CPU={row['cpu_ms']:9.3f} ms"
    )
    if row["gpu_ms"] is None:
        print(message + " | GPU=skipped")
        continue
    transfer = (
        f" | transfer+compute+copy-back={row['transfer_ms']:.3f} ms"
        if row["transfer_ms"] is not None
        else ""
    )
    print(
        message
        + f" | GPU={row['gpu_ms']:9.3f} ms"
        + f" | speedup={row['speedup']:6.2f}x"
        + f" | GPU={row['gpu_tflops']:.3f} TFLOP/s"
        + f" | max_error={row['max_error']:.3e}"
        + transfer
    )

## Explicit transfer experiment

This cell mirrors the lesson's two scenarios: compute with GPU-resident inputs, then transfer inputs to the GPU and copy the result back.

In [ ]:
if device.type != "cuda":
    print("Transfer experiment skipped: CUDA is not available.")
else:
    n = 2048
    transfer_repeats = 20
    generator = torch.Generator(device="cpu").manual_seed(17)
    A_cpu = torch.randn(n, n, generator=generator, dtype=torch.float32)
    B_cpu = torch.randn(n, n, generator=generator, dtype=torch.float32)
    A_gpu = A_cpu.to(device)
    B_gpu = B_cpu.to(device)

    # Compare the normal resident path with a complete CPU -> GPU -> CPU
    # round trip. Synchronization makes the host timer include GPU work.
    compute_seconds, _ = median_seconds(
        lambda: A_gpu @ B_gpu, transfer_repeats, device
    )
    roundtrip_samples = []
    for _ in range(transfer_repeats):
        # Prevent asynchronous work from the previous iteration from
        # leaking into this iteration's host-side timing sample.
        torch.cuda.synchronize(device)
        start_time = time.perf_counter()
        A = A_cpu.to(device)
        B = B_cpu.to(device)
        C_cpu = (A @ B).cpu()
        torch.cuda.synchronize(device)
        del A, B, C_cpu
        roundtrip_samples.append(time.perf_counter() - start_time)

    print(f"Compute only: {compute_seconds * 1000:.3f} ms")
    print(
        "Transfer + compute + return: "
        f"{statistics.median(roundtrip_samples) * 1000:.3f} ms"
    )

## LLM memory and batch-size experiment

This cell estimates 1B-parameter FP16 weight memory and measures how a projection changes when one token position is replaced by a larger batch.

In [ ]:
parameter_count = 1_000_000_000
# FP16 stores each parameter in two bytes; this excludes runtime memory.
bytes_per_fp16_parameter = torch.tensor([], dtype=torch.float16).element_size()
weight_bytes = parameter_count * bytes_per_fp16_parameter
print(f"1B parameters in FP16: {weight_bytes / 2**30:.2f} GiB")

hidden_size = 4096
projection_dtype = torch.float16 if device.type == "cuda" else torch.float32
W = torch.randn(hidden_size, hidden_size, device=device, dtype=projection_dtype)
print(f"Projection weights: {W.numel() * W.element_size() / 2**20:.2f} MiB")

for batch in (1, 64):
    # The weight matrix stays fixed while more token positions share it.
    X = torch.randn(batch, hidden_size, device=device, dtype=projection_dtype)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    elapsed, Y = median_seconds(lambda: X @ W, repeats, device)
    # A multiply plus an add is counted as approximately two FLOPs.
    flops = 2 * batch * hidden_size * hidden_size
    memory = (
        f" | peak={torch.cuda.max_memory_allocated(device) / 2**20:.1f} MiB"
        if device.type == "cuda"
        else ""
    )
    print(
        f"B={batch:2d} | X={tuple(X.shape)} | Y={tuple(Y.shape)} | "
        f"FLOPs={flops / 1e9:.3f}B | time={elapsed * 1000:.3f} ms"
        + memory
    )

## Plot the measured timings

The plot uses your actual measurements. A log-scaled y-axis makes it easier to compare CPU and GPU timings across matrix sizes.

In [ ]:
try:
    import matplotlib.pyplot as plt

    ns = [row["size"] for row in results]
    cpu_times = [row["cpu_ms"] for row in results]
    plt.figure(figsize=(8, 4))
    plt.plot(ns, cpu_times, marker="o", label="CPU")
    if device.type == "cuda":
        gpu_times = [row["gpu_ms"] for row in results]
        plt.plot(ns, gpu_times, marker="o", label="T4 GPU")
    plt.yscale("log")
    plt.xlabel("Square matrix dimension N")
    plt.ylabel("Median execution time (ms, log scale)")
    plt.title("CPU vs NVIDIA T4 — Matrix Multiplication")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()
except ImportError:
    print("Install matplotlib to plot the benchmark results.")

## Interpretation checklist

- At what matrix size does the GPU begin to amortize launch overhead?
- Does throughput improve as the matrix gets larger?
- How much extra latency does the transfer-inclusive path add?
- Is the maximum CPU/GPU error within an acceptable tolerance?
- How does this differ from a real LLM decode step, where the batch and matrix shapes may be much smaller?